## Disabling Grammar Restrictions in Reasoning Sections

When working with reasoning models that use special tokens like `<think>...</think>` to denote reasoning sections, you might want to allow free-form text within these sections while still enforcing grammar constraints on the rest of the output.

SGLang provides a feature to disable grammar restrictions within reasoning sections. This is particularly useful for models that need to perform complex reasoning steps before providing a structured output.

To enable this feature, use the `--reasoning-parser` flag which decide the think_end_token, such as `</think>`, when launching the server. You can also specify the reasoning parser using the `--reasoning-parser` flag.

## Supported Models

Currently, SGLang supports the following reasoning models:
- [DeepSeek R1 series](https://huggingface.co/collections/deepseek-ai/deepseek-r1-678e1e131c0169c0bc89728d): The reasoning content is wrapped with `<think>` and `</think>` tags.
- [QwQ](https://huggingface.co/Qwen/QwQ-32B): The reasoning content is wrapped with `<think>` and `</think>` tags.


## Usage

## OpenAI Compatible API

Specify the `--grammar-backend`, `--reasoning-parser` option.

In [1]:
import openai
import os
from sglang.test.test_utils import is_in_ci

if is_in_ci():
    from patch import launch_server_cmd
else:
    from sglang.utils import launch_server_cmd

from sglang.utils import wait_for_server, print_highlight, terminate_process

os.environ["TOKENIZERS_PARALLELISM"] = "false"


server_process, port = launch_server_cmd(
    "python -m sglang.launch_server --model-path /home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B --host 0.0.0.0 --grammar-backend xgrammar --reasoning-parser deepseek-r1"
)

wait_for_server(f"http://localhost:{port}")

INFO 03-29 19:15:08 __init__.py:190] Automatically detected platform cuda.
[2025-03-29 19:15:10] server_args=ServerArgs(model_path='/home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', tokenizer_path='/home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', tokenizer_mode='auto', skip_tokenizer_init=False, load_format='auto', trust_remote_code=False, dtype='auto', kv_cache_dtype='auto', quantization=None, quantization_param_path=None, context_length=None, device='cuda', served_model_name='/home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', chat_template=None, is_embedding=False, revision=None, host='0.0.0.0', port=31218, mem_fraction_static=0.88, max_running_requests=None, max_total_tokens=None, chunked_prefill_size=8192, max_prefill_tokens=16384, schedule_policy='fcfs', schedule_conservativeness=1.0, cpu_offload_gb=0, page_size=1, tp_size=1, stream_interval=1, stream_output=False, random

### JSON

you can directly define a JSON schema or use [Pydantic](https://docs.pydantic.dev/latest/) to define and validate the response.

**Using Pydantic**

In [2]:
import json

import openai
from pydantic import BaseModel, Field
client = openai.Client(base_url=f"http://127.0.0.1:{port}/v1", api_key="None")

# Define the schema using Pydantic
class CapitalInfo(BaseModel):
    name: str = Field(..., pattern=r"^\w+$", description="Name of the capital city")
    population: int = Field(..., description="Population of the capital city")


response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    messages=[
        {
            "role": "user",
            "content": "Please generate the information of the capital of France in the JSON format.",
        },
    ],
    temperature=0,
    max_tokens=2048,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "foo",
            # convert the pydantic model to json schema
            "schema": CapitalInfo.model_json_schema(),
        },
    },
)

response_content = response.choices[0].message.content
print(response.choices[0].message.content)
print(response.choices[0].message.reasoning_content)

# validate the JSON response by the pydantic model
# capital_info = CapitalInfo.model_validate_json(response_content)
# print(f"Validated response: {capital_info.model_dump_json()}")

[2025-03-29 19:15:41 TP0] Prefill batch. #new-seq: 1, #new-token: 18, #cached-token: 1, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:15:41 TP0] Decode batch. #running-req: 1, #token: 52, token usage: 0.00, gen throughput (token/s): 2.23, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:15:42 TP0] Decode batch. #running-req: 1, #token: 92, token usage: 0.00, gen throughput (token/s): 65.36, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:15:42 TP0] Decode batch. #running-req: 1, #token: 132, token usage: 0.00, gen throughput (token/s): 65.31, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:15:43 TP0] Decode batch. #running-req: 1, #token: 172, token usage: 0.00, gen throughput (token/s): 65.22, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:15:44 TP0] Decode batch. #running-req: 1, #token: 212, token usage: 0.00, gen throughput (token/s): 65.21, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:15:44 TP0] Decode batch. #running-req: 1, #token: 252, token usage: 0.00, gen 

In [3]:
# validate the JSON response by the pydantic model
capital_info = CapitalInfo.model_validate_json(response_content)
print(f"Validated response: {capital_info.model_dump_json()}")

ValidationError: 1 validation error for CapitalInfo
  Invalid JSON: EOF while parsing an object at line 4 column 1140 [type=json_invalid, input_value='{\n\n"name": "Paris",\n"...00000000000000000000000', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid

**JSON Schema Directly**


In [11]:
import json

json_schema = json.dumps(
    {
        "type": "object",
        "properties": {
            "name": {"type": "string", "pattern": "^[\\w]+$"},
            "population": {"type": "integer"},
        },
        "required": ["name", "population"],
    }
)

response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    messages=[
        {
            "role": "user",
            "content": "Give me the information of the capital of France in the JSON format.",
        },
    ],
    temperature=0,
    max_tokens=1024,
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "foo", "schema": json.loads(json_schema)},
    },
)

print(response.choices[0].message.content)

[2025-03-29 19:13:03 TP0] Prefill batch. #new-seq: 1, #new-token: 1, #cached-token: 18, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:13:03 TP0] Decode batch. #running-req: 1, #token: 23, token usage: 0.00, gen throughput (token/s): 0.26, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:13:03 TP0] Decode batch. #running-req: 1, #token: 63, token usage: 0.00, gen throughput (token/s): 65.44, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:13:04 TP0] Decode batch. #running-req: 1, #token: 103, token usage: 0.00, gen throughput (token/s): 65.36, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:13:05 TP0] Decode batch. #running-req: 1, #token: 143, token usage: 0.00, gen throughput (token/s): 65.34, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:13:05 TP0] Decode batch. #running-req: 1, #token: 183, token usage: 0.00, gen throughput (token/s): 65.27, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:13:06 TP0] Decode batch. #running-req: 1, #token: 223, token usage: 0.00, gen 

### EBNF

In [5]:
ebnf_grammar = """
root ::= city | description
city ::= "London" | "Paris" | "Berlin" | "Rome"
description ::= city " is " status
status ::= "the capital of " country
country ::= "England" | "France" | "Germany" | "Italy"
"""

response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    messages=[
        {"role": "system", "content": "You are a helpful geography bot."},
        {
            "role": "user",
            "content": "Give me the information of the capital of France.",
        },
    ],
    temperature=0,
    max_tokens=1024,
    extra_body={"ebnf": ebnf_grammar},
)
print(response)
print(response.choices[0].message.content)

[2025-03-29 19:16:54 TP0] Prefill batch. #new-seq: 1, #new-token: 1, #cached-token: 21, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:16:54 TP0] Decode batch. #running-req: 1, #token: 35, token usage: 0.00, gen throughput (token/s): 2.26, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:16:55 TP0] Decode batch. #running-req: 1, #token: 75, token usage: 0.00, gen throughput (token/s): 65.38, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:16:55 TP0] Decode batch. #running-req: 1, #token: 115, token usage: 0.00, gen throughput (token/s): 65.34, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:16:56 TP0] Decode batch. #running-req: 1, #token: 155, token usage: 0.00, gen throughput (token/s): 65.27, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:16:56] INFO:     127.0.0.1:33730 - "POST /v1/chat/completions HTTP/1.1" 200 OK
ChatCompletion(id='4309cd6075284662b2c68f4a27fdcf87', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content

### Regular expression

In [8]:
response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    messages=[
        {"role": "user", "content": "What is the capital of France?"},
    ],
    temperature=0,
    max_tokens=2048,
    extra_body={"regex": "(Paris|London)"},
)
print(response)
# print(f"{response.choices[0].message.content}")

[2025-03-29 19:17:40 TP0] Prefill batch. #new-seq: 1, #new-token: 1, #cached-token: 11, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:17:40 TP0] Decode batch. #running-req: 1, #token: 32, token usage: 0.00, gen throughput (token/s): 2.25, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:17:41 TP0] Decode batch. #running-req: 1, #token: 72, token usage: 0.00, gen throughput (token/s): 65.45, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:17:41 TP0] Decode batch. #running-req: 1, #token: 112, token usage: 0.00, gen throughput (token/s): 65.32, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:17:42 TP0] Decode batch. #running-req: 1, #token: 152, token usage: 0.00, gen throughput (token/s): 65.28, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:17:42] INFO:     127.0.0.1:60558 - "POST /v1/chat/completions HTTP/1.1" 200 OK
ChatCompletion(id='69b47a2e7be94e138bc59fdaee1b2e55', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content

## Native API and SGLang Runtime (SRT)

### JSON

**Using Pydantic**

In [18]:
import requests
import json
from pydantic import BaseModel, Field

from transformers import AutoTokenizer
# Define the schema using Pydantic
class CapitalInfo(BaseModel):
    name: str = Field(..., pattern=r"^\w+$", description="Name of the capital city")
    population: int = Field(..., description="Population of the capital city")

messages = [
    {
        "role": "user",
        "content": "Here is the information of the capital of France in the JSON format.\n",
     }
]
tokenizer = AutoTokenizer.from_pretrained("/home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# Make API request
response = requests.post(
    f"http://localhost:{port}/generate",
    json={
        "text": text,
        "sampling_params": {
            "temperature": 0,
            "max_new_tokens": 4096,
            "json_schema": json.dumps(CapitalInfo.model_json_schema()),
        },
    },
)
print(response.json())


response_data = json.loads(response.json()["text"].split("</think>")[1])
# validate the response by the pydantic model
capital_info = CapitalInfo.model_validate(response_data)
print(f"Validated response: {capital_info.model_dump_json()}")

[2025-03-29 19:26:06 TP0] Prefill batch. #new-seq: 1, #new-token: 1, #cached-token: 19, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:26:06 TP0] Decode batch. #running-req: 1, #token: 55, token usage: 0.00, gen throughput (token/s): 0.27, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:26:07 TP0] Decode batch. #running-req: 1, #token: 95, token usage: 0.00, gen throughput (token/s): 65.39, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:26:07 TP0] Decode batch. #running-req: 1, #token: 135, token usage: 0.00, gen throughput (token/s): 65.35, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:26:08 TP0] Decode batch. #running-req: 1, #token: 175, token usage: 0.00, gen throughput (token/s): 65.25, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:26:09 TP0] Decode batch. #running-req: 1, #token: 215, token usage: 0.00, gen throughput (token/s): 65.24, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:26:09 TP0] Decode batch. #running-req: 1, #token: 255, token usage: 0.00, gen 

**JSON Schema Directly**

In [20]:
json_schema = json.dumps(
    {
        "type": "object",
        "properties": {
            "name": {"type": "string", "pattern": "^[\\w]+$"},
            "population": {"type": "integer"},
        },
        "required": ["name", "population"],
    }
)

# JSON
response = requests.post(
    f"http://localhost:{port}/generate",
    json={
        "text": "Here is the information of the capital of France in the JSON format.\n",
        "sampling_params": {
            "temperature": 0,
            "max_new_tokens": 1024,
            "json_schema": json_schema,
        },
    },
)

print(response.json())

[2025-03-29 19:28:03 TP0] Prefill batch. #new-seq: 1, #new-token: 1, #cached-token: 14, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:28:04 TP0] Decode batch. #running-req: 1, #token: 52, token usage: 0.00, gen throughput (token/s): 0.57, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:04 TP0] Decode batch. #running-req: 1, #token: 92, token usage: 0.00, gen throughput (token/s): 65.37, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:05 TP0] Decode batch. #running-req: 1, #token: 132, token usage: 0.00, gen throughput (token/s): 65.36, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:06 TP0] Decode batch. #running-req: 1, #token: 172, token usage: 0.00, gen throughput (token/s): 65.23, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:06 TP0] Decode batch. #running-req: 1, #token: 212, token usage: 0.00, gen throughput (token/s): 65.22, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:07 TP0] Decode batch. #running-req: 1, #token: 252, token usage: 0.00, gen 

### EBNF

In [21]:
import requests

response = requests.post(
    f"http://localhost:{port}/generate",
    json={
        "text": "Give me the information of the capital of France.",
        "sampling_params": {
            "max_new_tokens": 128,
            "temperature": 0,
            "n": 3,
            "ebnf": (
                "root ::= city | description\n"
                'city ::= "London" | "Paris" | "Berlin" | "Rome"\n'
                'description ::= city " is " status\n'
                'status ::= "the capital of " country\n'
                'country ::= "England" | "France" | "Germany" | "Italy"'
            ),
        },
        "stream": False,
        "return_logprob": False,
    },
)

print(response.json())

[2025-03-29 19:28:41 TP0] Prefill batch. #new-seq: 1, #new-token: 10, #cached-token: 1, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:28:41 TP0] Prefill batch. #new-seq: 3, #new-token: 3, #cached-token: 30, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:28:42 TP0] Decode batch. #running-req: 3, #token: 47, token usage: 0.00, gen throughput (token/s): 2.68, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:42 TP0] Decode batch. #running-req: 3, #token: 167, token usage: 0.00, gen throughput (token/s): 188.82, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:43 TP0] Decode batch. #running-req: 3, #token: 287, token usage: 0.00, gen throughput (token/s): 188.67, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:28:43] INFO:     127.0.0.1:52954 - "POST /generate HTTP/1.1" 200 OK
[{'text': "600 words.\n\nThe capital of France is Paris. Paris is one of the most important cities in the world, and it's also the political, cultural, and economic center of 

### Regular expression

In [22]:
response = requests.post(
    f"http://localhost:{port}/generate",
    json={
        "text": "Paris is the capital of",
        "sampling_params": {
            "temperature": 0,
            "max_new_tokens": 64,
            "regex": "(France|England)",
        },
    },
)
print(response.json())

[2025-03-29 19:29:10 TP0] Prefill batch. #new-seq: 1, #new-token: 5, #cached-token: 1, token usage: 0.00, #running-req: 0, #queue-req: 0, 
[2025-03-29 19:29:10 TP0] Decode batch. #running-req: 1, #token: 10, token usage: 0.00, gen throughput (token/s): 4.21, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:29:11 TP0] Decode batch. #running-req: 1, #token: 50, token usage: 0.00, gen throughput (token/s): 65.40, largest-len: 0, #queue-req: 0, 
[2025-03-29 19:29:11] INFO:     127.0.0.1:44976 - "POST /generate HTTP/1.1" 200 OK
{'text': ' the \\( n \\912121212121212121212121212121212121212121212121212121212121', 'meta_info': {'id': '05d37fe14a6f4db2afeaf885b4c20d0b', 'finish_reason': {'type': 'length', 'length': 64}, 'prompt_tokens': 6, 'completion_tokens': 64, 'cached_tokens': 1, 'e2e_latency': 1.0108082294464111}}


In [ ]:
terminate_process(server_process)

## Offline Engine API

In [1]:
import sglang as sgl

llm = sgl.Engine(
    model_path="/home/tianhaoyu/.cache/modelscope/hub/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    grammar_backend="xgrammar",
    reasoning_parser="deepseek-r1"
)

INFO 03-29 19:30:30 __init__.py:190] Automatically detected platform cuda.
INFO 03-29 19:30:35 __init__.py:190] Automatically detected platform cuda.
INFO 03-29 19:30:35 __init__.py:190] Automatically detected platform cuda.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.04it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.18s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:02<00:00,  1.15s/it]

Capturing batches (avail_mem=2.51 GB): 100%|██████████| 23/23 [00:04<00:00,  5.43it/s]


### JSON

**Using Pydantic**

In [2]:
import json
from pydantic import BaseModel, Field


prompts = [
    "Give me the information of the capital of China in the JSON format.",
    "Give me the information of the capital of France in the JSON format.",
    "Give me the information of the capital of Ireland in the JSON format.",
]


# Define the schema using Pydantic
class CapitalInfo(BaseModel):
    name: str = Field(..., pattern=r"^\w+$", description="Name of the capital city")
    population: int = Field(..., description="Population of the capital city")


sampling_params = {
    "temperature": 0.1,
    "top_p": 0.95,
    "json_schema": json.dumps(CapitalInfo.model_json_schema()),
}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}")  # validate the output by the pydantic model
    capital_info = CapitalInfo.model_validate_json(output["text"])
    print(f"Validated output: {capital_info.model_dump_json()}")

Prompt: Give me the information of the capital of China in the JSON format.


ValidationError: 1 validation error for CapitalInfo
  Invalid JSON: expected value at line 1 column 2 [type=json_invalid, input_value=' and also, make sure tha...y necessary corrections', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid

In [4]:
outputs

[{'text': ' and also, make sure that the JSON is valid.\n\n```json\n{\n  "name": "Beijing",\n  "population": 10000000,\n  "area": 100000,\n  "founded": 1500,\n  "coordinates": {\n    "latitude": "40.4168",\n    "longitude": "-74.0802"\n  }\n}\n```\n\nIs this JSON valid? If not, explain why.\n\nIf it is valid, explain why.\n\nAlso, provide an updated version of the JSON with any necessary corrections',
  'meta_info': {'id': '8f968f438c554e09b6d8359717478d0d',
   'finish_reason': {'type': 'length', 'length': 128},
   'prompt_tokens': 15,
   'completion_tokens': 128,
   'cached_tokens': 0,
   'e2e_latency': 2.871372938156128}},
 {'text': " and then convert that JSON into a JSONP object.\n\nAlso, create a JSON array containing 5 different cities in France, each with their population and area.\n\nFinally, create a JSON object that contains both the information of the capital and the array of 5 cities.\n\nMake sure to include all the required fields: for the city info, it's name, population,

**JSON Schema Directly**

In [ ]:
prompts = [
    "Give me the information of the capital of China in the JSON format.",
    "Give me the information of the capital of France in the JSON format.",
    "Give me the information of the capital of Ireland in the JSON format.",
]

json_schema = json.dumps(
    {
        "type": "object",
        "properties": {
            "name": {"type": "string", "pattern": "^[\\w]+$"},
            "population": {"type": "integer"},
        },
        "required": ["name", "population"],
    }
)

sampling_params = {"temperature": 0.1, "top_p": 0.95, "json_schema": json_schema}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}\nGenerated text: {output['text']}")

### EBNF


In [ ]:
prompts = [
    "Give me the information of the capital of France.",
    "Give me the information of the capital of Germany.",
    "Give me the information of the capital of Italy.",
]

sampling_params = {
    "temperature": 0.8,
    "top_p": 0.95,
    "ebnf": (
        "root ::= city | description\n"
        'city ::= "London" | "Paris" | "Berlin" | "Rome"\n'
        'description ::= city " is " status\n'
        'status ::= "the capital of " country\n'
        'country ::= "England" | "France" | "Germany" | "Italy"'
    ),
}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}\nGenerated text: {output['text']}")

### Regular expression

In [ ]:
prompts = [
    "Please provide information about London as a major global city:",
    "Please provide information about Paris as a major global city:",
]

sampling_params = {"temperature": 0.8, "top_p": 0.95, "regex": "(France|England)"}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}\nGenerated text: {output['text']}")

In [ ]:
llm.shutdown()